# KazLLM-8B — проверка на RunPod A40

`issai/LLama-3.1-KazLLM-1.0-8B` — национальная модель Казахстана от ISSAI (тот же институт, что делает Qolda),
стандартная архитектура Llama 3.1, без custom code. Заявленные языки в карточке — **kk, en, ru, tr**
(на один язык больше, чем у Sherkala — там только kk/en).

Зрения у модели нет — сравнение чисто текстовое, третье и последнее в серии после
[01_qolda_check.ipynb](01_qolda_check.ipynb) и [02_sherkala_check.ipynb](02_sherkala_check.ipynb).

Лицензия — `cc-by-nc-4.0` (некоммерческая, как и у Sherkala) — учесть при решении про продакшн.

Репозиторий `gated: auto` — те же шаги, что для Sherkala: запросить доступ на странице модели, создать HF-токен.

In [ ]:
# --- 0. Проверка GPU ---
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram total GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
    print('vram free GB:', round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 2**30, 1))
else:
    raise SystemExit('GPU недоступна в этом контейнере.')

In [ ]:
# --- HF-токен для gated-репозитория ---
# Тот же токен, что использовался для Sherkala (huggingface.co/settings/tokens),
# при условии что доступ к issai/LLama-3.1-KazLLM-1.0-8B тоже запрошен на странице модели.
# НЕ коммить этот токен в git.
import os

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = 'hf_ВСТАВЬ_СВОЙ_ТОКЕН_СЮДА'

# .strip() убирает случайные пробелы/переводы строк при вставке (в т.ч. из Secrets-хранилищ),
# а проверка на чистый ASCII ловит невидимые символы, из-за которых HTTP-заголовок ломается
# с UnicodeEncodeError — именно это уже случалось при первой попытке.
token = os.environ['HF_TOKEN'].strip()
try:
    token.encode('ascii')
except UnicodeEncodeError as e:
    raise ValueError(
        f'Токен содержит не-ASCII символы ({e}). Скопируй его заново с huggingface.co/settings/tokens, '
        'проверь что не зацепился лишний символ до/после.'
    )
os.environ['HF_TOKEN'] = token

from huggingface_hub import login, whoami
login(token=token)

me = whoami(token=token)
print('HF login OK, user:', me.get('name'))

In [ ]:
# --- Кэш в /workspace (переживает рестарт пода, / — всего 5GB) ---
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.makedirs('/workspace/hf_cache', exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# --- Зависимости ---
# Та же версия, что и для Sherkala/Qolda — для честного сравнения на одинаковом стеке.
%pip install -q "transformers==4.55.0" accelerate sentencepiece hf_transfer

In [ ]:
import transformers
print('transformers version:', transformers.__version__)

## Загрузка модели

`AutoModelForCausalLM` + `AutoTokenizer`, без `trust_remote_code` — как и Sherkala, стандартная Llama.

In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'issai/LLama-3.1-KazLLM-1.0-8B'

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).eval().cuda()
print(f'Загружено за {time.time()-t0:.1f}s')
print('VRAM занято, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

In [ ]:
print('chat_template встроен:', tokenizer.chat_template is not None)
if tokenizer.chat_template:
    print(tokenizer.chat_template[:200], '...')

## Тест 1 — языки (kk/ru/en)

Те же промпты, что в предыдущих двух ноутбуках — для прямого сравнения бок о бок по трём моделям.

In [ ]:
GEN_KW = dict(max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)

def ask_text(question):
    messages = [{'role': 'user', 'content': question}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    ).cuda()
    with torch.no_grad():
        out = model.generate(input_ids, **GEN_KW)
    new_tokens = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

prompts = {
    'kk': 'Қазақстанның бас қаласы қандай және неге астана болып таңдалды? Қысқаша жауап бер.',
    'ru': 'Объясни своими словами, чем отличается ИП от ТОО в Казахстане. Кратко.',
    'en': 'In two sentences, explain what a vector database is used for in RAG systems.',
}

for lang, q in prompts.items():
    t0 = time.time()
    answer = ask_text(q)
    dt = time.time() - t0
    print(f'=== {lang} ({dt:.1f}s) ===')
    print('Q:', q)
    print('A:', answer)
    print()

**На что смотреть:** KazLLM заявляет русский официально (в отличие от Sherkala) — ожидаем, что `ru`-ответ будет увереннее. Плюс это модель того же института (ISSAI), что и Qolda — интересно, различается ли терминология/стиль казахского между двумя их моделями.

## Тест 2 — tool calling (быстрый сигнал, не входит в scope этапа)

In [ ]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Получить текущую погоду в городе',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string', 'description': 'Название города'}},
            'required': ['city'],
        },
    },
}]

messages = [{'role': 'user', 'content': 'Қандай ауа райы Алматыда қазір?'}]

try:
    input_ids = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True, return_tensors='pt'
    ).cuda()
    with torch.no_grad():
        out = model.generate(input_ids, **GEN_KW)
    new_tokens = out[0][input_ids.shape[-1]:]
    tool_answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(tool_answer)
except Exception as e:
    print('Chat template не принимает tools= напрямую:', e)

## Замер VRAM

In [ ]:
print('VRAM allocated, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))
print('VRAM reserved,  GB:', round(torch.cuda.memory_reserved() / 2**30, 2))
print('VRAM peak,      GB:', round(torch.cuda.max_memory_allocated() / 2**30, 2))

In [ ]:
# Освободить память, если на этом поде дальше запускается что-то ещё.
del model
torch.cuda.empty_cache()
print('VRAM после очистки, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

## Дальше по плану

Три модели прогнаны на одинаковых промптах — свести результаты (языки, зрение только у Qolda, VRAM, latency, tool calling)
в сравнительную таблицу в `docs/findings.md`. На этом стенд сравнения из шага 2 плана закрыт; дальше — решение,
какая модель идёт в каркас `app/`, и (если нужно) сравнение bf16 против Q4_K_M для локального железа.